# WeatherGPT trained models — one cell per model

This notebook loads the 5 models trained for **WeatherGPT** (SIH 2026) directly from the public Hugging Face repo [`Arko007/weathergpt-models`](https://huggingface.co/Arko007/weathergpt-models) — no token needed, it's public. Each model gets **its own cell**, so you can run just the one you care about after the setup cell.

| # | model | what it does |
|---|---|---|
| M1 | field mapper | maps a raw provider field name (`APCP`, `2t`, `tp`, ...) to a canonical variable |
| M2 | MOS bias correction | corrects a multi-model NWP forecast against ERA5, with a calibrated interval |
| M3 | intent + slot parser | extracts intent + LOCATION/TIME/CROP spans from a natural-language weather question, in 13 Indian languages + English |
| M4 | precipitation calibration | turns a multi-model ensemble spread into a verified exceedance probability at each rain threshold |
| M5 | trust ranker | ranks which NWP source (GFS/ECMWF/ICON/GEM) to trust for a given variable, location and lead time |

**Every model has an admission gate**: if its `metrics.json` can't prove it beats the baseline it replaces, the corresponding `registry.<name>` attribute is `None` and the cell below says so instead of crashing — that's intentional, not a bug. See `registry.status()` in the setup cell for the honest reason behind every load/refuse decision.

Full technical detail (real held-out numbers, and a separate real-world/live-data usability check) lives in the source repo: `docs/MODEL_REGISTRY_INTEGRATION.md` and `docs/REAL_WORLD_READINESS.md` at [`Anamitra-Sarkar/weathergpt`](https://github.com/Anamitra-Sarkar/weathergpt).

## Setup — install, clone the inference package, load the registry

`weathergpt_models` isn't on PyPI (yet) — it's a plain package inside the WeatherGPT repo with no external dependency on the rest of that repo (no FastAPI, no `app/` import), so we just clone the repo and add it to `sys.path`. The actual model **weights** come from Hugging Face, not from this clone.

In [ ]:
!pip install -q torch transformers==4.46.3 tokenizers==0.20.3 huggingface_hub lightgbm scikit-learn scipy
!rm -rf /kaggle/working/weathergpt
!git clone --depth 1 https://github.com/Anamitra-Sarkar/weathergpt.git /kaggle/working/weathergpt

import sys
sys.path.insert(0, "/kaggle/working/weathergpt")

from weathergpt_models import ModelRegistry

# Public repo -> no HF token needed. This downloads ~2.16 GB once and caches it.
registry = ModelRegistry.from_hub("Arko007/weathergpt-models")

print("Load status for all 5 models (loaded=True/False and why):\n")
for entry in registry.status():
    print(f"  {entry['model']:14s} loaded={entry['loaded']!s:6s} {entry['reason']}")

---
## M1 — field mapper

Maps a raw field name from *any* weather data provider (NOAA GFS `.idx` names, ECMWF short names, WRF registry names, IMD product names, ...) to one of 46 canonical variables the rest of the pipeline understands. Trained on CF/GRIB2/CAP parameter tables, tested zero-shot on WRF/NCEP tables it never saw — held-out macro-F1 **0.743** vs. **0.160** for a hand-written dict lookup (4.6x). If it isn't confident, it abstains rather than guessing — never trust a mapping it didn't return.

In [ ]:
if registry.field_mapper is None:
    print("M1 did not load -- see registry.status() above for why.")
else:
    # A mix of real field names the model has and hasn't seen labelled before.
    examples = [
        ("APCP", "kg m-2", "Total precipitation"),          # NOAA GFS
        ("2t", "K", "2 metre temperature"),                   # ECMWF short name
        ("tp", "m", "Total precipitation"),                   # ECMWF short name
        ("10si", "m s**-1", "10 metre wind speed"),           # ECMWF short name
        ("PWAT_entire_atmosphere", "kg m-2", "Precipitable water"),  # genuinely obscure
    ]
    for name, unit, description in examples:
        mapping = registry.field_mapper.map_field(name, unit=unit, description=description)
        if mapping.is_usable:
            print(f"{name:24s} -> {mapping.canonical_variable:24s} (confidence {mapping.confidence:.2f})")
        else:
            print(f"{name:24s} -> ABSTAIN (confidence {mapping.confidence:.2f}) -- do not fabricate a mapping here")

---
## M2 — MOS bias correction

Corrects a 4-model NWP forecast (GFS + ECMWF + ICON + GEM) against ERA5 truth: a LightGBM quantile forest blended with a monotone quantile network, conformalised for a real 80% interval. Spatially held-out CRPS skill vs. the raw ensemble: **+4.7%** temperature, **+13.2%** precipitation, **+8.9%** wind. Known edge case: under-represented regimes (e.g. very low wind in an arid location) can produce a correction outside the raw forecasts' own range — sanity-check the interval against the raw spread before showing it to a user.

In [ ]:
if registry.mos is None:
    print("M2 did not load -- see registry.status() above for why.")
else:
    forecasts = {"gfs_seamless": 31.2, "ecmwf_ifs025": 30.4, "icon_seamless": 30.9, "gem_seamless": 31.8}
    context = {"lead_hours": 30, "lead_age_days": 1, "hour_utc": 6, "doy": 245,
               "elevation_m": 303.0, "lat": 21.15, "lon": 79.09}

    corrected = registry.mos.correct("temperature_2m", forecasts=forecasts, context=context)
    raw_mean = sum(forecasts.values()) / len(forecasts)

    print(f"raw 4-model mean:     {raw_mean:.2f} C")
    print(f"corrected (M2):       {corrected.value:.2f} C  (correction {corrected.correction:+.2f})")
    print(f"80% interval:         [{corrected.interval_low:.2f}, {corrected.interval_high:.2f}] C")
    print(f"full quantiles:       {corrected.quantiles}")

---
## M3 — intent + slot parser

JointBERT-style head on `google/muril-base-cased` (17 Indian languages + transliterations, so Hinglish/Banglish are in-distribution). Held-out intent macro-F1 **0.746** vs. **0.166** for the rule-based parser it augments (4.5x); slot F1 **0.988**.

**Honest caveat, found by testing on live never-templated queries, not just held-out data**: `intent_confidence` on real casual phrasing runs **0.15-0.19** — barely above the ~0.125 uniform-guess floor for 8 classes. Treat `intent` as a weak signal to combine with a rule-based parser, not a standalone decision. LOCATION/TIME slot extraction held up much better on the same live queries (10/12 correct through typos and shorthand) and is safer to lean on. Never use `.variables` (macro-F1 0.12, most of the 46 canonical variables still have thin support).

In [ ]:
if registry.intent is None:
    print("M3 did not load -- see registry.status() above for why.")
else:
    queries = [
        "kal Bhandara me baarish hogi kya?",
        "Should I spray pesticide on my cotton in Yavatmal tomorrow?",
        "Can fishermen go to sea off Ratnagiri tomorrow?",
        "is it gonna rain today in puri or what",   # genuinely casual, never templated
    ]
    for query in queries:
        parsed = registry.intent.parse(query)
        print(f"query:      {query}")
        print(f"  intent:   {parsed.intent}  (confidence {parsed.intent_confidence:.3f})")
        print(f"  LOC span: {parsed.slot_text('LOC')}")
        print(f"  TIME span:{parsed.slot_text('TIME')}")
        print()

---
## M4 — precipitation calibration

A hurdle model (explicit P(wet) x censored shifted-gamma) fitted by minimising CRPS, refined per threshold by isotonic regression against real observed frequency. Calibrated Brier score beats raw ensemble member-counting at every one of 6 thresholds (0.1/1/5/10/25/50 mm). This is the highest-value integration point for RADE-style decisions: it replaces naive "X out of N members say it rains" with a verified probability.

In [ ]:
if registry.calibration is None:
    print("M4 did not load -- see registry.status() above for why.")
else:
    forecasts = {"gfs_seamless": 4.2, "ecmwf_ifs025": 1.1, "icon_seamless": 6.8, "gem_seamless": 0.0}
    context = {"lead_hours": 30, "lead_age_days": 1, "hour_utc": 6, "doy": 245,
               "elevation_m": 303.0, "lat": 21.15, "lon": 79.09}

    curve = registry.calibration.probability_curve("precipitation", forecasts=forecasts, context=context)
    for point in curve:
        print(f"P(precip > {point.threshold:5.1f} mm) = {point.probability:.3f}  "
              f"(raw member frequency {point.raw_ensemble_frequency:.3f})")

---
## M5 — trust ranker

LightGBM LambdaMART, ranking the 4 trained NWP sources (GFS/ECMWF/ICON/GEM) by *learned* trust for a given variable, location, season and lead time -- replacing a single fixed authority table with one that can say "ECMWF for temperature here, GFS for wind there". Beats the fixed authority order on NDCG@1 for all 3 trained variables (temperature +9.3% RMSE skill, precipitation +6.0%, wind +12.8%).

In [ ]:
if registry.trust_ranker is None:
    print("M5 did not load -- see registry.status() above for why.")
else:
    candidates = {"gfs_seamless": 31.2, "ecmwf_ifs025": 30.4, "icon_seamless": 30.9, "gem_seamless": 31.8}
    context = {"lead_hours": 30, "season": "autumn", "elevation_m": 303.0, "lat": 21.15, "lon": 79.09}

    ranked = registry.trust_ranker.rank("temperature_2m", candidates=candidates, context=context)
    print("ranked best-to-worst source for temperature_2m at this location/season/lead:")
    for r in ranked:
        print(f"  #{r.rank}  {r.source:14s} score={r.score:+.2f}  value={r.value}")

---
## Where to go next

- Every metric quoted above came from a real `metrics.json` on the Hugging Face repo — see `Arko007/weathergpt-models`'s model card for the full held-out tables and per-language breakdown for M3.
- For wiring these into an actual FastAPI service (which exact file/function to touch, what to never change), see `docs/MODEL_REGISTRY_INTEGRATION.md` in [`Anamitra-Sarkar/weathergpt`](https://github.com/Anamitra-Sarkar/weathergpt).
- For the live-data / real-user-phrasing usability findings (including the M3 confidence caveat demonstrated above), see `docs/REAL_WORLD_READINESS.md` in the same repo.